# Land-Cover Classification from Satellite Images (EuroSAT) with CNNs

**Goal:** classify 64x64 Sentinel-2 satellite image patches into 10 land-use / land-cover classes (forest, river, highway, residential, ...).

**Pipeline**

1. Load and explore the EuroSAT dataset
2. Stratified train / validation / test split with augmentation
3. Baseline CNN trained from scratch
4. Transfer learning with a pretrained ResNet18
5. Evaluation: accuracy, per-class metrics, confusion matrix, most-confused classes
6. Explainability with Grad-CAM
7. Save the model so the FastAPI service in `api/main.py` can serve predictions

> **Tip:** run on a GPU (Colab: *Runtime > Change runtime type > T4 GPU*). The baseline trains in a few minutes on a GPU.
>
> **Note:** the notebook downloads EuroSAT (about 90 MB) through `torchvision` the first time it runs.

In [ ]:
# Run once in a fresh environment (Colab already has most of these). Uncomment if needed.
# %pip install -q torch torchvision scikit-learn matplotlib seaborn pandas fastapi uvicorn python-multipart pillow

In [ ]:
import os
import random
import time
import copy
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models

SEED = 42


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

## 1. Configuration and data

In [ ]:
# Works whether the notebook is opened from the repo root or from the notebooks/ folder
ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
DATA_DIR = os.path.join(ROOT, "data")
MODEL_DIR = os.path.join(ROOT, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

BATCH_SIZE = 64
NUM_WORKERS = 2
BASELINE_EPOCHS = 15
TRANSFER_EPOCHS = 6

In [ ]:
# Load the raw dataset (PIL images, no transforms yet). Downloads on first run.
raw = datasets.EuroSAT(root=DATA_DIR, download=True)

CLASS_NAMES = raw.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"{len(raw)} images, {NUM_CLASSES} classes")
print(CLASS_NAMES)

In [ ]:
# Class distribution
targets = np.array(raw.targets)
counts = pd.Series(targets).value_counts().sort_index()
counts.index = [CLASS_NAMES[i] for i in counts.index]

plt.figure(figsize=(8, 4))
counts.sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Number of images")
plt.title("EuroSAT class distribution")
plt.tight_layout()
plt.show()

In [ ]:
# Stratified 70 / 15 / 15 split so every class is represented in each subset
idx = np.arange(len(targets))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=targets, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=targets[temp_idx], random_state=SEED)
print(f"train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}")

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


class TransformSubset(Dataset):
    """Applies a transform to a subset of the raw dataset (different transforms per split)."""

    def __init__(self, base, indices, transform):
        self.base = base
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, label = self.base[int(self.indices[i])]
        return self.transform(img), label


def make_loaders(img_size=64):
    train_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    pin = DEVICE.type == "cuda"
    train_ds = TransformSubset(raw, train_idx, train_tf)
    val_ds = TransformSubset(raw, val_idx, eval_tf)
    test_ds = TransformSubset(raw, test_idx, eval_tf)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
    return train_loader, val_loader, test_loader


def denormalize(t):
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (t * std + mean).clamp(0, 1)


train_loader, val_loader, test_loader = make_loaders(64)

In [ ]:
# Look at a batch of (augmented) training images
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, img, lab in zip(axes.flat, images, labels):
    ax.imshow(denormalize(img).permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[lab], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Baseline CNN (trained from scratch)

Four convolution blocks (Conv - BatchNorm - ReLU - MaxPool), global average pooling, dropout, and a linear classifier.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(c_out),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.4), nn.Linear(256, num_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def count_params(model):
    return sum(p.numel() for p in model.parameters())


baseline = SimpleCNN(NUM_CLASSES).to(DEVICE)
print(f"Baseline CNN parameters: {count_params(baseline):,}")

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over a loader. Trains if an optimizer is given, otherwise evaluates."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss = criterion(out, y)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, val_loader, epochs, optimizer, scheduler=None, name="model"):
    """Trains the model and restores the weights of the best validation epoch."""
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = run_epoch(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        history["train_loss"].append(tr_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(tr_acc)
        history["val_acc"].append(val_acc)
        if val_acc > best_acc:
            best_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"[{name}] epoch {epoch:02d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} "
              f"| val loss {val_loss:.4f} acc {val_acc:.4f} | {time.time() - t0:.0f}s")
    model.load_state_dict(best_state)
    return history, best_acc


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="validation")
    axes[0].set_title(f"{title}: loss")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="validation")
    axes[1].set_title(f"{title}: accuracy")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
optimizer = torch.optim.AdamW(baseline.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=BASELINE_EPOCHS)

baseline_hist, baseline_best_val = fit(baseline, train_loader, val_loader, BASELINE_EPOCHS,
                                       optimizer, scheduler, name="baseline")
torch.save(baseline.state_dict(), os.path.join(MODEL_DIR, "simple_cnn.pt"))
print(f"\nBest validation accuracy: {baseline_best_val:.4f}")

In [ ]:
plot_history(baseline_hist, "Baseline CNN")

## 3. Evaluation on the held-out test set

In [ ]:
@torch.no_grad()
def predict_all(model, loader):
    model.eval()
    preds, trues = [], []
    for x, y in loader:
        out = model(x.to(DEVICE))
        preds.append(out.argmax(1).cpu())
        trues.append(y)
    return torch.cat(trues).numpy(), torch.cat(preds).numpy()


def evaluate(model, loader, title):
    y_true, y_pred = predict_all(model, loader)
    acc = accuracy_score(y_true, y_pred)
    print(f"{title} - test accuracy: {acc:.4f}\n")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{title}: confusion matrix (test set)")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
    return acc, cm


def top_confusions(cm, k=5):
    pairs = []
    for i in range(len(cm)):
        for j in range(len(cm)):
            if i != j and cm[i, j] > 0:
                pairs.append((int(cm[i, j]), CLASS_NAMES[i], CLASS_NAMES[j]))
    pairs.sort(reverse=True)
    print(f"Top {k} confusions (true class -> predicted as):")
    for count, true_name, pred_name in pairs[:k]:
        print(f"  {true_name} -> {pred_name}: {count} images")


baseline_acc, baseline_cm = evaluate(baseline, test_loader, "Baseline CNN")
top_confusions(baseline_cm)

## 4. Transfer learning with ResNet18

A ResNet18 pretrained on ImageNet is fine-tuned on EuroSAT. Images are upscaled to 128x128 so the pretrained filters see more spatial detail. The backbone uses a smaller learning rate than the new classification head.

In [ ]:
IMG_SIZE_TL = 128
tl_train, tl_val, tl_test = make_loaders(IMG_SIZE_TL)

weights = models.ResNet18_Weights.DEFAULT  # downloads ImageNet weights on first run
resnet = models.resnet18(weights=weights)
resnet.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(resnet.fc.in_features, NUM_CLASSES))
resnet = resnet.to(DEVICE)
print(f"ResNet18 parameters: {count_params(resnet):,}")

head_params = list(resnet.fc.parameters())
backbone_params = [p for n, p in resnet.named_parameters() if not n.startswith("fc.")]
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": 1e-4}, {"params": head_params, "lr": 1e-3}],
    weight_decay=1e-4,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TRANSFER_EPOCHS)

resnet_hist, resnet_best_val = fit(resnet, tl_train, tl_val, TRANSFER_EPOCHS,
                                   optimizer, scheduler, name="resnet18")
torch.save(resnet.state_dict(), os.path.join(MODEL_DIR, "resnet18_eurosat.pt"))
with open(os.path.join(MODEL_DIR, "class_names.json"), "w") as f:
    json.dump(CLASS_NAMES, f)
print(f"\nBest validation accuracy: {resnet_best_val:.4f}")

In [ ]:
plot_history(resnet_hist, "ResNet18 (fine-tuned)")
resnet_acc, resnet_cm = evaluate(resnet, tl_test, "ResNet18 (fine-tuned)")
top_confusions(resnet_cm)

## 5. Model comparison

In [ ]:
results = pd.DataFrame({
    "Model": ["Baseline CNN (from scratch)", "ResNet18 (fine-tuned)"],
    "Parameters": [count_params(baseline), count_params(resnet)],
    "Test accuracy": [round(baseline_acc, 4), round(resnet_acc, 4)],
})
results["Parameters"] = results["Parameters"].map("{:,}".format)
results

## 6. Explainability with Grad-CAM

Grad-CAM highlights the image regions that pushed the model toward its prediction. It is computed from the last convolutional block of ResNet18.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)

    def _save_activation(self, module, inputs, output):
        self.activations = output.detach()
        output.register_hook(self._save_gradient)

    def _save_gradient(self, grad):
        self.gradients = grad.detach()

    def __call__(self, x, class_idx=None):
        self.model.zero_grad()
        out = self.model(x)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        probs = F.softmax(out, dim=1)[0].detach().cpu()
        return cam[0, 0].cpu().numpy(), class_idx, probs


cam_model = GradCAM(resnet, resnet.layer4[-1])
tl_test_ds = tl_test.dataset

sample_ids = random.sample(range(len(tl_test_ds)), 8)
fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for k, i in enumerate(sample_ids):
    x, y = tl_test_ds[i]
    cam, pred, probs = cam_model(x.unsqueeze(0).to(DEVICE))
    img = denormalize(x).permute(1, 2, 0).numpy()
    axes[0, k].imshow(img)
    axes[0, k].set_title(f"true: {CLASS_NAMES[y]}", fontsize=8)
    axes[1, k].imshow(img)
    axes[1, k].imshow(cam, cmap="jet", alpha=0.45)
    axes[1, k].set_title(f"pred: {CLASS_NAMES[pred]} ({probs[pred]:.0%})", fontsize=8)
    axes[0, k].axis("off")
    axes[1, k].axis("off")
plt.tight_layout()
plt.show()

## 7. Serve the model with FastAPI

Training saved `models/resnet18_eurosat.pt` and `models/class_names.json`. The API in `api/main.py` loads them and exposes:

- `GET /health` - service status
- `POST /predict` - upload a JPEG or PNG satellite patch and get the predicted land-cover class with the top-3 probabilities

From the repository root, in a terminal:

```
uvicorn api.main:app --reload
```

Then open http://127.0.0.1:8000/docs for the interactive Swagger UI, or call it with curl:

```
curl -X POST http://127.0.0.1:8000/predict -F "file=@tile.png;type=image/png"
```

In [ ]:
# Confirm the artifacts the API needs were saved
for name in sorted(os.listdir(MODEL_DIR)):
    path = os.path.join(MODEL_DIR, name)
    print(f"{name:25s} {os.path.getsize(path) / 1e6:6.1f} MB")

## 8. Results and next steps

Fill in your own numbers after running the notebook, then copy them into the README and your resume:

| Model | Parameters | Test accuracy |
|---|---|---|
| Baseline CNN | *(from the table above)* | *(from the table above)* |
| ResNet18 (fine-tuned) | *(from the table above)* | *(from the table above)* |

**README checklist**

- One-paragraph summary and the comparison table
- Confusion matrix image and the most-confused class pairs, with a short explanation of why
- A few Grad-CAM examples
- How to run: install requirements, run the notebook, then `uvicorn api.main:app --reload`
- Honest limitations: 64x64 patches, European locations only, accuracy may drop on other regions

**Ideas to extend it**

- Try more architectures (MobileNetV3, EfficientNet) and compare accuracy against model size
- Containerize the API with Docker
- Test on imagery from another region to measure how well the model generalizes
- Add test-time augmentation or a class-weighted loss for the weakest classes